# LightGBM: The Drastic Solution
Unlike Neural Networks, Gradient Boosted Trees do not require `StandardScaler` or `RobustScaler`. They completely ignore numeric scale and sparsity, which means they mathematically bypass the exact roadblock that destroyed TabNet, TabPFN, Keras, MLP, and KNN.

This is the exact architecture the original researchers used to achieve 99% accuracy on EMBER.

In [ ]:
import os
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

print("LightGBM successfully imported!")

In [ ]:
# --- 1. LOAD DATA USING MEMMAP ---
DATASET_DIR = r"Z:\ember2024_train_data"
ndim = 2381
nrows_to_read = 200000  # We can load a massive chunk now because LightGBM is highly optimized

X_path = os.path.join(DATASET_DIR, "X_train.dat")
y_path = os.path.join(DATASET_DIR, "y_train.dat")

print("Reading massive chunk from disk...")
X_memmap = np.memmap(X_path, dtype=np.float32, mode="r", shape=(nrows_to_read, ndim))
y_memmap = np.memmap(y_path, dtype=np.int32, mode="r", shape=(nrows_to_read,))

# Convert to in-RAM numpy array to speed up target filtering
X_chunk = np.array(X_memmap)
y_chunk = np.array(y_memmap)

# Filter out unlabeled (-1)
valid_idx = np.where(y_chunk != -1)[0]
X_chunk = X_chunk[valid_idx]
y_chunk = y_chunk[valid_idx]

print(f"Extracted {len(y_chunk)} Total Labeled Files.")

print("Splitting into Train / Validation (80/20)...")
X_train, X_valid, y_train, y_valid = train_test_split(
    X_chunk, y_chunk, test_size=0.2, random_state=42, stratify=y_chunk
)

del X_chunk
del y_chunk
import gc
gc.collect()

print(f"Train Shape: {X_train.shape} | Valid Shape: {X_valid.shape}")

### Notice: NO SCALING!
We are **omitting** `RobustScaler` and `StandardScaler` entirely. We feed the raw byte metrics straight into the Trees.

In [ ]:
# --- 2. TRAIN LIGHTGBM ---
print("Initializing LightGBM (Gradient Boosted Trees)...")
train_data = lgb.Dataset(X_train, label=y_train)
valid_data = lgb.Dataset(X_valid, label=y_valid, reference=train_data)

params = {
    'objective': 'binary',
    'metric': ['auc', 'binary_error'],
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 256,
    'max_depth': -1,
    'feature_fraction': 0.6,    # Sub-sample features to prevent overfitting noise
    'n_jobs': -1                # Use all CPU Cores
}

print("Starting Model Training...")
# This will train remarkably fast and display progress every 20 iterations
model = lgb.train(
    params, 
    train_data, 
    num_boost_round=1000, 
    valid_sets=[train_data, valid_data], 
    callbacks=[
        lgb.early_stopping(stopping_rounds=50),
        lgb.log_evaluation(period=20)
    ]
)

In [ ]:
# --- 3. EVALUATE FINAL TEST ACCURACY ---
print("\nEvaluating Accuracy and ROC_AUC...")
y_prob = model.predict(X_valid)
y_pred = (y_prob > 0.5).astype(int)

acc = accuracy_score(y_valid, y_pred)
auc = roc_auc_score(y_valid, y_prob)

print(f"\n===== [ LIGHTGBM DECISION TREES ] =====")
print(f"Final Accuracy: {acc*100:.2f}%")
print(f"Final AUC:      {auc:.4f}")